# Sinusoidal Positional Encoding — Exercise

Companion to [Positional Encoding Part 1 § Sinusoidal](https://tanulsingh.github.io/blog/positional-encoding-absolute#sinusoidal-positional-encoding-vaswani-et-al-2017).

You'll build the sinusoidal positional encoding from Vaswani et al. (2017) and verify its rotation property numerically.

$$PE_{(pos,\, 2i)} = \sin\left(\frac{pos}{10000^{2i/d_{\text{model}}}}\right)$$

$$PE_{(pos,\, 2i+1)} = \cos\left(\frac{pos}{10000^{2i/d_{\text{model}}}}\right)$$

## Setup

In [4]:
# TODO: imports — torch, torch.nn as nn, math, matplotlib

## TODO 1 — Frequencies per dimension pair

From the blog: $\omega_i = \frac{1}{10000^{2i/d_{\text{model}}}}$ for $i = 0, 1, \ldots, d_{\text{model}}/2 - 1$.

Low $i$ → high frequency (fast clock). High $i$ → low frequency (slow clock).

In [ ]:
def get_frequencies(d_model: int) -> 'torch.Tensor':
    """Return shape (d_model // 2,) tensor of frequencies omega_i."""
    # TODO 1: implement
    pass

In [ ]:
# Sanity check:
#   - get_frequencies(4) returns 2 values
#   - The first value is 1.0 (highest frequency)
#   - Values decrease as i increases

## TODO 2 — Build the PE matrix

Return a `(max_len, d_model)` matrix where:
- $PE[pos, 2i] = \sin(\omega_i \cdot pos)$
- $PE[pos, 2i+1] = \cos(\omega_i \cdot pos)$

In [ ]:
def sinusoidal_pe(max_len: int, d_model: int) -> 'torch.Tensor':
    """Return shape (max_len, d_model) sinusoidal positional encoding."""
    # TODO 2: implement
    pass

In [ ]:
# Sanity check:
#   - shape == (max_len, d_model)
#   - PE[0] has all sines == 0 and all cosines == 1
#   - All values are in [-1, 1]

## TODO 3 — Verify the rotation property numerically

From the blog: there exists a fixed matrix $R_k$ such that $PE(pos+k) = R_k \cdot PE(pos)$.

$R_k$ is block-diagonal: one $2\times 2$ rotation per dimension pair, with rotation angle $\omega_i \cdot k$.

Pick `pos=10`, `k=5`, build $R_k$, and check that $R_k \cdot PE[10] \approx PE[15]$.

In [ ]:
# TODO 3:
#   - Build R_k as a block-diagonal matrix of 2x2 rotations
#   - Compute lhs = R_k @ PE[pos]
#   - Compute rhs = PE[pos + k]
#   - Assert torch.allclose(lhs, rhs, atol=1e-5)
#   - Try a few different (pos, k) pairs

## TODO 4 — Wrap it into a Module

Now bundle everything into a reusable `nn.Module`. This is what you'd actually plug into a transformer.

The interface should match `LearnedPositionalEmbedding` (next notebook), so a transformer can swap between them with no other code changes:

- `__init__(max_len, d_model)` — pre-compute the PE matrix once at construction
- `forward(token_embeddings)` — take `(batch, seq_len, d_model)` and return it with PE added

**Key design question:** should the PE matrix be a `nn.Parameter` or something else? Sinusoidal PE has no learnable values — there's nothing to train. Look up `register_buffer` to see how PyTorch handles fixed tensors that should still move with `.to(device)` and save with `state_dict()`.

In [ ]:
class SinusoidalPositionalEmbedding(nn.Module):
    def __init__(self, max_len: int, d_model: int):
        super().__init__()
        # TODO 4a: pre-compute the PE matrix using sinusoidal_pe(max_len, d_model)
        # TODO 4b: register it as a buffer (NOT a parameter — it's not learnable)
        pass

    def forward(self, token_embeddings):
        # token_embeddings: (batch, seq_len, d_model)
        # TODO 4c: slice PE to seq_len and add to token_embeddings (broadcasting over batch)
        # Return shape: (batch, seq_len, d_model)
        pass

In [ ]:
# Sanity check:
#   - module = SinusoidalPositionalEmbedding(max_len=128, d_model=64)
#   - sum(p.numel() for p in module.parameters()) == 0  (no learnable params!)
#   - forward on a (2, 10, 64) tensor returns shape (2, 10, 64)
#   - The buffer name (e.g. 'pe') appears in module.state_dict()

## Run the tests

In [ ]:
from tests import run_sinusoidal_tests, run_sinusoidal_module_tests
# run_sinusoidal_tests(sinusoidal_pe)
# run_sinusoidal_module_tests(SinusoidalPositionalEmbedding)